# Massvis Fixation Iidividual level analysis

Created on 20210401

@author: Yao Wang

@usage: Bucket fixations with durations and element bounding boxes, and generate a string to represent the scanpath. Format supported for both GT and baseline methods.

In [31]:
# -*- coding: utf-8 -*-
import numpy as np
import os, sys
from PIL import Image
from natsort import natsorted
from glob import glob
import pandas as pd
from tqdm import trange
from skimage.draw import polygon
from polygon_util import bucketColorsNew
np.set_printoptions(suppress=True)

# change base path to the directory where you have downloaded the salicon data
bp = "/path/to/Massvis/images"
curdir = '../../eyetracking/csv_files/fixationsByVis'
eledir = '../../dataset/csv_files/targets393_elementLabels/elementLabels/'
origimdir = os.path.join(bp, 'targets')
curfiles = natsorted(glob(curdir + '/*'))

indiv_names = ['abl','cmb','lcm','ls','mg','pi','rb','rk','sok', 'wab']
indiv_name = indiv_names[6]
indiv_cnt = {}
BucketStr=['A','X','G','L','O','T','S','D','Z']


class BBox:
    name = ''
    coords = None
    id = -1
    sm = '_'
    durations = None
    def __init__(self, name, coords, id):
        self.name = name
        self.coords = coords
        self.id = id
        self.sm = BucketStr[id]

# Bucket 1-D index for semantic regions:
# 0 A: Annotation
# 1 X: Axis
# 2 G: Graphical Element
# 3 L: Legend
# 4 O: Object [Photograph, Pictogram]
# 5 T: Text [Title, Header Row, Paragraph]
# 6 S: Text [Label, Source]
# 7 D: Data
# 8 Z: Other 

# Bucket 2-D index for Durations:
# 0: <500ms
# 1: 500-3000ms
# 2: 3000-5000ms
# 3: 5000-7500ms
# 4: 7500ms+


def get_bboxid_by_name(name):
    id1 = -1
    if 'annotation' in name:
        id1 = 0
    elif 'axis' in name:
        id1 = 1
    elif 'graphical element' in name:
        id1 = 2
    elif 'legend' in name:
        id1 = 3
    elif 'object' in name:
        id1 = 4
    elif 'text' in name:
        if '(title)' in name or '(header row)' in name or '(paragraph)' in name:
            id1 = 5
        else:
            id1 = 6
    elif 'data' in name:
        id1 = 7
    else:
        id1 = 8
    return id1

def get_accum_bucket(accum_time):
    accum_time = int(accum_time)
    if accum_time < 1000:
        id2 = 0
    elif accum_time < 2000:
        id2 = 1
    elif accum_time < 3000:
        id2 = 2
    elif accum_time < 4000:
        id2 = 3
    elif accum_time < 5000:
        id2 = 4
    elif accum_time < 6000:
        id2 = 5
    elif accum_time < 7000:
        id2 = 6
    elif accum_time < 8000:
        id2 = 7
    elif accum_time < 9000:
        id2 = 8
    else:
        id2 = -1
    return id2

mds_bucket = np.zeros((9, 9))
dur_bucket = np.zeros((600))

### Stats for individual samples

In [32]:
indiv_cnt = {}
for id in trange(0,len(curfiles), 1):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)

    # load the raw image
    with Image.open(os.path.join(origimdir, imname+'.png')) as im:
        width, height = im.size # original image size
        allfiles = glob(os.path.join(curfiles[id], 'enc' ,'*.csv'))

        for subcsv in allfiles:
            name = subcsv.split('/')[-1]
            if name in indiv_cnt:
                indiv_cnt[name] += 1
            else:
                indiv_cnt[name] = 1

print(indiv_cnt)

100%|██████████| 393/393 [00:00<00:00, 8223.10it/s]

{'rb.csv': 393, 'rk.csv': 393, 'sok.csv': 393, 'abl.csv': 294, 'em.csv': 195, 'rs.csv': 98, 'md.csv': 197, 'sh.csv': 98, 'mc.csv': 294, 'ad.csv': 98, 'ls.csv': 294, 'pk.csv': 196, 'db.csv': 196, 'lcm.csv': 393, 'mg.csv': 393, 'tj.csv': 98, 'pi.csv': 393, 'cmb.csv': 393, 'vm.csv': 98, 'wab.csv': 343, 'am.csv': 98, 'mrc.csv': 98, 'ch.csv': 98, 'sk.csv': 98, 'ag.csv': 99, 'zg.csv': 98, 'jm.csv': 98, 'pj.csv': 98, 'dn.csv': 98, 'ra.csv': 98, 'cy.csv': 98, 'krm.csv': 98, 'ms.csv': 98, 'lg.csv': 25}


### Read Individual scanpath, stat fixation and convert to String

In [33]:
outpath = os.path.join(bp, 'plot_individual', indiv_name)
#strpath = "../../eyetracking/csv_files/strings"
strpath = "../../eyetracking/csv_files/strings_5000"

os.makedirs(outpath, exist_ok=True)

mds_bucket = np.zeros((9, 9))


#for id in trange(len(curfiles)-1,-1,-1):
for id in trange(0,len(curfiles),1):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)


    # load the raw image
    with Image.open(os.path.join(origimdir, imname+'.png')) as im:
        width, height = im.size # original image size

        #Handle Overlap problem
        id_map = np.full([width,height], 8)

        for tt in range(7, -1, -1):
            #print(tt)
            for box in boxes:
                if box.id == tt:
                    rr, cc = polygon(np.array(box.coords)[:,0], np.array(box.coords)[:,1], (width, height))
                    id_map[rr, cc] = tt


        subcsv = os.path.join(curfiles[id], 'enc' ,indiv_name + '.csv')

        if os.path.exists(subcsv):
            scanpath_str = ""
            #Add the image to the path
            fig = plt.figure(figsize=(16,16))

            plt.imshow(im)

            fixations = pd.read_csv(subcsv, header=None, names=['index','x','y','time'])

            x = fixations['x'].tolist()
            y = fixations['y'].tolist()
            duration = fixations['time'].tolist()

            accum_time = 0
            for row2 in fixations.iterrows():
                accum_time += row2[1][3]
                if (int(row2[0]) < 2): continue
                # If the fixation is inside the box, add one to the bucket counter
                xx = int(row2[1][1])
                yy = int(row2[1][2])

                s_roi = BucketStr[id_map[xx][yy]]


                id1 = id_map[xx, yy]
                id2 = get_accum_bucket(accum_time)
                # Truncate 5s
                if id2 < 5: scanpath_str += s_roi
                #scanpath_str += s_roi
                if id1 >= 0 and id2 < 9 and id2 >= 0 and id2 < 9:
                    mds_bucket[id1][id2] += 1
                if(row2[1][3]<600):
                    dur_bucket[int(row2[1][3])] += 1
            # save scanpath string
            fp = '%s/%s/%s_%s.txt'%(strpath, indiv_name, imname, indiv_name)
            file = open(fp,'w');
            file.write(str(scanpath_str));
            file.close();

            # expell the first fixation
            #x.pop(0)
            #y.pop(0)
            #duration.pop(0)

            plt.plot(x,y, '-ro', color='yellow',lw=1)
            #visualize the plot on image
            for i in range(len(x)):
                #if(x[i] > width): print('error')
                #if(y[i] > height): print('error')
                dur = duration[i]/10 if duration[i]/10>10 else 10
                dur = dur if dur<20 else 20
                if(i == 0):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.5) ) # 
                elif(i == len(x) - 1):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.2), mec='#e9b96e')
                else:
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.5), mec='#e9b96e') 
                plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

            for i in range(len(elementX)):
                plt.fill(elementX[i], elementY[i], color=bucketColorsNew(boxes[i].id))

            plt.title(imname+indiv_name)
            fig.savefig('%s/%s_%s.png'%(outpath, imname, indiv_name), dpi=fig.dpi)
            plt.close()

print(mds_bucket)

np.savez('%s.npz'%(indiv_name),dur_bucket)

100%|██████████| 393/393 [03:37<00:00,  1.80it/s]

[[ 12.  34.  41.  77.  76.  62.  68.  47.  49.]
 [ 47. 156. 167. 206. 226. 262. 195. 188. 122.]
 [ 23.  53.  45.  22.  20.  12.  26.  18.  18.]
 [ 31.  95. 102. 131. 127. 125. 108.  79.  61.]
 [ 36.  80.  87.  82.  98. 111. 120. 109. 102.]
 [341. 663. 494. 371. 258. 269. 200. 170. 115.]
 [ 40. 132. 160. 182. 205. 200. 191. 212. 151.]
 [117. 278. 367. 409. 457. 405. 433. 395. 339.]
 [192. 321. 221. 170. 133. 101.  95.  84.  42.]]


### Display Element fixation percentage

In [70]:
res = mds_bucket.copy()
for i in range(np.shape(res)[1]):
    res[:,i] /= np.sum(res[:,i])
    res[:,i] = np.round(100*res[:,i], 2)

res

array([[ 3.22,  3.02,  4.28,  3.88,  3.37,  4.99,  5.07,  4.74,  3.82],
       [ 3.04,  7.92, 11.56, 13.63, 15.56, 11.76, 11.46, 11.71,  9.89],
       [ 2.55,  2.57,  1.67,  1.24,  1.01,  1.63,  1.09,  0.62,  2.25],
       [ 2.74,  5.54,  7.67,  8.7 ,  6.85,  7.74,  6.44,  6.36,  8.09],
       [ 7.85,  5.84,  6.15,  6.71,  7.65,  7.48,  8.57,  7.91,  8.43],
       [27.62, 37.85, 25.43, 18.  , 16.26, 15.48, 14.62, 16.07, 17.75],
       [ 4.81,  6.33,  8.71, 11.93, 11.03, 13.39, 12.55, 11.34, 12.36],
       [39.05, 19.2 , 26.51, 28.64, 29.56, 29.07, 33.99, 32.96, 30.79],
       [ 9.12, 11.73,  8.02,  7.26,  8.71,  8.45,  6.22,  8.29,  6.63]])

## Analysis of Beseline Methods

In [7]:
METHOD_NAME='PathGAN_finetuned'
#METHOD_NAME='PathGAN_unfinetuned'

mds_bucket = np.zeros((14, 20))

outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
predpath = os.path.join(bp, METHOD_NAME, 'scanpaths')
os.makedirs(outpath, exist_ok=True)

SUBJECTS = 25
imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)



    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,imname+'.png.csv'))
        df_pred.columns = ['user', 'x', 'y', 'time']

        #Add the image to the path
        fig = plt.figure(figsize=(16,16))
        plt.imshow(im)

        #for pp in range(SUBJECTS):
        for pp in range(1):
            df_predTmp = df_pred[df_pred['user'] == pp]
            df_predI = df_predTmp.drop(['user'] ,axis=1)
    
            x = df_predI['x'].tolist()
            y = df_predI['y'].tolist()
            duration = df_predI['time'].tolist()

            accum_time = 0
            for row2 in df_predI.iterrows():
                accum_time += row2[1][2]
                if (int(row[0]) < 1): continue
                for box in boxes:
                    if isPoiWithinPoly([int(row2[1][0]),int(row2[1][1])], box.coords):
                        id2 = get_accum_bucket(accum_time)
                        mds_bucket[box.id][id2] += 1

            # expell the first fixation
            #x.pop(0)
            #y.pop(0)
            #duration.pop(0)

            plt.plot(x,y, '-ro', color='yellow',lw=2)
            #visualize the plot on image
            for i in range(len(x)):
                dur = duration[i]/10 if duration[i]/10>10 else 10
                dur = dur if dur<20 else 20
                if(i == 0):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
                elif(i == len(x) - 1):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
                else:
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
                plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColorsNew(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:21<00:00,  3.12it/s]


In [5]:
METHOD_NAME='nc2020'

mds_bucket = np.zeros((14, 20))

outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
predpath = os.path.join(bp, METHOD_NAME, 'tsv')
os.makedirs(outpath, exist_ok=True)

imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)


    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,'ncnc_'+imname+'.tsv'), sep='\t')
        df_pred.columns = ['x', 'y', 'time']

        #Add the image to the path
        fig = plt.figure(figsize=(16,16))

        plt.imshow(im)

        x = df_pred['x'].tolist()
        y = df_pred['y'].tolist()
        duration = df_pred['time'].tolist()

        accum_time = 0
        for row2 in df_pred.iterrows():
            accum_time += row2[1][2]
            if (int(row[0]) < 1): continue
            for box in boxes:
                if isPoiWithinPoly([int(row2[1][0]),int(row2[1][1])], box.coords):
                    id2 = get_accum_bucket(accum_time)
                    mds_bucket[box.id][id2] += 1
                    
        # expell the last (-1,-1,-1) fixation
        x.pop(-1)
        y.pop(-1)
        duration.pop(-1)

        plt.plot(x,y, '-ro', color='yellow',lw=2)
        #visualize the plot on image
        for i in range(len(x)):
            dur = duration[i]/10 if duration[i]/10>10 else 10
            dur = dur if dur<20 else 20

            if(i == 0):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
            elif(i == len(x) - 1):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
            else:
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
            plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColorsNew(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:16<00:00,  3.89it/s]


In [8]:
METHOD_NAME='PathGAN_finetuned_10'

predpath = os.path.join(bp, METHOD_NAME, 'tsv')
outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
os.makedirs(outpath, exist_ok=True)

imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)


    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,imname+'.png.tsv'), header=None, sep='\t')
        df_pred.columns = ['x', 'y', 'time']

        #Add the image to the path
        fig = plt.figure(figsize=(16,16))

        plt.imshow(im)

        x = df_pred['x'].tolist()
        y = df_pred['y'].tolist()
        duration = df_pred['time'].tolist()

        accum_time = 0
        for row2 in df_pred.iterrows():
            accum_time += row2[1][2]
            if (int(row[0]) < 1): continue
            for box in boxes:
                if isPoiWithinPoly([int(row2[1][0]),int(row2[1][1])], box.coords):
                    id2 = get_accum_bucket(accum_time)
                    mds_bucket[box.id][id2] += 1
                    
        # expell the last (-1,-1,-1) fixation
        x.pop(-1)
        y.pop(-1)
        duration.pop(-1)

        plt.plot(x,y, '-ro', color='yellow',lw=2)
        #visualize the plot on image
        for i in range(len(x)):
            dur = duration[i]/10 if duration[i]/10>10 else 10
            dur = dur if dur<20 else 20

            if(i == 0):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
            elif(i == len(x) - 1):
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
            else:
                plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
            plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColorsNew(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:17<00:00,  3.79it/s]


In [3]:
METHOD_NAME='Saltinet'

mds_bucket = np.zeros((14, 20))

predpath = os.path.join(bp, METHOD_NAME, 'scanpaths')
outpath = os.path.join(bp, METHOD_NAME, 'plot_all_fixations')
os.makedirs(outpath, exist_ok=True)

SUBJECTS = 25
imgfiles = natsorted(glob(origimdir + '/*.png'))

for id in trange(0, len(imgfiles), 6):
    basepath = os.path.basename(curfiles[id])
    imname, ext = os.path.splitext(basepath)
    #print('imname',imname)
    elementLabel = pd.read_csv(os.path.join(eledir, imname), names=['bboxID','category','x','y'])
    elementCoords = []
    elementX = []
    elementY = []
    tmp = 0
    curName = ""

    boxes = []

    for row in elementLabel.iterrows():
        # row[1][0]: id
        # row[1][1]: category name
        # row[1][2]: x
        # row[1][3]: y

        # a new bbox
        if int(row[1][0]) > tmp:
            # Store the last bbox
            if tmp > 0:
                boxid = get_bboxid_by_name(curName)
                box = BBox(curName, elementCoords[-1], boxid)
                boxes.append(box)
            tmp = int(row[1][0])
            curName = row[1][1].strip()

            elementCoords.append([])
            elementX.append([])
            elementY.append([])

        elementCoords[-1].append([int(row[1][2]), int(row[1][3])])
        if(curName != 'data'):
            elementX[-1].append(int(row[1][2]))
            elementY[-1].append(int(row[1][3]))

    boxid = get_bboxid_by_name(curName)
    box = BBox(curName, elementCoords[-1], boxid)
    boxes.append(box)



    # load the raw image
    with Image.open(imgfiles[id]) as im:
        width, height = im.size # original image size

        df_pred = pd.read_csv(os.path.join(predpath,imname+'.png.csv'))
        df_pred.columns = ['user', 'index', 'time', 'x', 'y']
        #Add the image to the path
        fig = plt.figure(figsize=(16,16))
        plt.imshow(im)

        #for pp in range(SUBJECTS):
        for pp in range(2):
            df_predTmp = df_pred[df_pred['user'] == pp + 1]
            df_predI = df_predTmp.drop(['user','index'] ,axis=1)

            df_predI['time'] *= 1000
            x = df_predI['x'].tolist()
            y = df_predI['y'].tolist()
            duration = df_predI['time'].tolist()

            accum_time = 0
            for row2 in df_predI.iterrows():
                accum_time += row2[1][0]
                if (int(row[0]) < 1): continue
                for box in boxes:
                    if isPoiWithinPoly([int(row2[1][1]),int(row2[1][2])], box.coords):
                        id2 = get_accum_bucket(accum_time)
                        mds_bucket[box.id][id2] += 1

            # expell the first fixation
            #x.pop(0)
            #y.pop(0)
            #duration.pop(0)

            plt.plot(x,y, '-ro', color='yellow',lw=2)
            #visualize the plot on image
            for i in range(len(x)):
                dur = duration[i]/10 if duration[i]/10>10 else 10
                dur = dur if dur<20 else 20
                if(i == 0):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.1, 0.1, 1.0, 0.7))
                elif(i == len(x) - 1):
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(0.5, 0.5, 0.5, 0.7), mec='#e9b96e')
                else:
                    plt.plot(x[i], y[i], '-o', ms = dur, mfc=(1.0, 0.0, 0.0, 0.7), mec='#e9b96e') 
                plt.annotate(i, (x[i], y[i]), color=(1, 1, 0, 0.7),weight='bold',fontsize=10)

        for i in range(len(elementX)):
            plt.fill(elementX[i], elementY[i], color=bucketColorsNew(boxes[i].id))

        plt.title('%s_%s_total'%(imname, METHOD_NAME))
        fig.savefig('%s/%s_sum.png'%(outpath, imname), dpi=fig.dpi)
        plt.close()


100%|██████████| 66/66 [00:24<00:00,  2.71it/s]
